# ECON 3916: ML Prediction Project — Final Project

**From Question to Recommendation**

This notebook scaffolds your final project. Work through each part sequentially. By Week 12, this notebook (plus your `app.py` and report) will form your complete submission.

**AI Policy:** AI co-pilot is REQUIRED. Document every AI interaction in Part 7 (AI Methodology Appendix) using the P.R.I.M.E. framework.

---

## Part 0: Setup

In [ ]:
# ============================================================
# Part 0: Setup — Run this cell first
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score, roc_curve
)

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

print('Setup complete.')

---
## Part 1: Problem Statement

**My prediction question is:** Given in-game team performance statistics (field goal percentage, 3-point percentage, free throw percentage, rebounds, and assists), can we predict whether the home team will win an NBA game?

**This is a prediction (not causation) problem because:** We are using observed in-game statistics to forecast a binary outcome (home win / away win). We make no claim that shooting a higher field goal percentage *causes* winning — many confounding factors (opponent quality, game context, player matchups, officiating) operate simultaneously. Our goal is accurate forecasting, not causal identification.

**The decision this enables:** This analysis would help a **sports analytics firm or betting platform** decide how to set real-time win-probability estimates for NBA games using live in-game box-score data as inputs.

**Dataset:** NBA Games Dataset
- **Source:** Kaggle — nathanlauga (URL: https://www.kaggle.com/datasets/nathanlauga/nba-games)
- **Files:** `games.csv` (primary), `games_details.csv`, `teams.csv`, `players.csv`, `ranking.csv`
- **N =** 26,552 games after dropping 99 rows with missing statistics (2003–2022 seasons)
- **Features =** 15 (10 raw team statistics + 5 engineered differential features)
- **Target variable =** `HOME_TEAM_WINS` (binary: 1 = home team wins, 0 = away team wins)
- **Access date:** April 21, 2026

---
## Part 2: Data Loading + EDA

### 2.1 Load Your Data

In [ ]:
# ============================================================
# 2.1 Load the NBA Games dataset
# Source: https://www.kaggle.com/datasets/nathanlauga/nba-games
# ============================================================

# Primary modeling file — one row per game
df = pd.read_csv('games.csv')

# Additional files available for extended analysis:
# games_details = pd.read_csv('games_details.csv', low_memory=False)  # player-level box scores
# teams         = pd.read_csv('teams.csv')                            # team metadata
# players       = pd.read_csv('players.csv')                          # player metadata
# rankings      = pd.read_csv('ranking.csv')                          # season standings

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# ── Drop rows missing core statistics (99 rows, 0.37%) ───────
stat_cols = ['PTS_home','FG_PCT_home','FT_PCT_home','FG3_PCT_home','AST_home','REB_home',
             'PTS_away','FG_PCT_away','FT_PCT_away','FG3_PCT_away','AST_away','REB_away']

df = df.dropna(subset=stat_cols).reset_index(drop=True)

# ── Feature Engineering ───────────────────────────────────────
# Differential features capture the competitive gap between teams
df['FG_PCT_DIFF']  = df['FG_PCT_home']  - df['FG_PCT_away']
df['FG3_PCT_DIFF'] = df['FG3_PCT_home'] - df['FG3_PCT_away']
df['FT_PCT_DIFF']  = df['FT_PCT_home']  - df['FT_PCT_away']
df['REB_DIFF']     = df['REB_home']     - df['REB_away']
df['AST_DIFF']     = df['AST_home']     - df['AST_away']

print(f'Clean shape: {df.shape}')
print(f'Seasons covered: {sorted(df["SEASON"].unique())}')
print(f'Home win rate:   {df["HOME_TEAM_WINS"].mean():.3f}')

### 2.2 Basic Description

In [ ]:
# ============================================================
# 2.2 Describe the data
# ============================================================

print('=== Data Types & Non-Null Counts ===')
df.info()
print('\n=== Summary Statistics (key columns) ===')
df[stat_cols + ['HOME_TEAM_WINS']].describe().round(3)

### 2.3 Missing Data Assessment

In [ ]:
# ============================================================
# 2.3 Missing data — assessed on raw file before dropping (Ch 1)
# ============================================================

df_raw = pd.read_csv('games.csv')
missing_pct = df_raw.isnull().mean().sort_values(ascending=False)
print('Missing data (%) by column — raw file:')
print(missing_pct[missing_pct > 0].to_string())
print(f'\nRows with any missing stat: '
      f'{df_raw[stat_cols].isnull().any(axis=1).sum()} / {len(df_raw)} '
      f'({df_raw[stat_cols].isnull().any(axis=1).mean()*100:.2f}%)')

# Visual: missing data heatmap
plt.figure(figsize=(12, 4))
sns.heatmap(df_raw[stat_cols + ['HOME_TEAM_WINS']].isnull(),
            cbar=True, yticklabels=False, cmap='viridis')
plt.title('Missing Data Heatmap — games.csv (raw, before cleaning)', fontsize=13)
plt.tight_layout()
plt.show()

**Missing data strategy:**

The raw `games.csv` has 99 rows (0.37%) where all 12 box-score statistic columns are missing simultaneously — but `GAME_ID` and `HOME_TEAM_WINS` are present. This complete-row missingness pattern indicates these are suspended or postponed games (e.g., COVID-19 bubble, weather postponements) rather than random data entry gaps.

Under the Ch. 1 MCAR/MAR/MNAR framework: this is **MCAR (Missing Completely At Random)** — the probability of a game having missing stats is unrelated to what those stats would have been. The game status (suspended, postponed) is what drives the missingness, not the statistics themselves.

**Strategy: listwise deletion.** With only 0.37% of rows affected and an MCAR mechanism, dropping these rows introduces no systematic bias. We proceed with N = 26,552 complete records.

### 2.4 Distribution Plots

In [ ]:
# ============================================================
# Visualization 1: Feature & target distributions (Ch 3)
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Home FG% distribution
sns.histplot(df['FG_PCT_home'], kde=True, ax=axes[0], color='steelblue', bins=40)
axes[0].axvline(df['FG_PCT_home'].mean(), color='red', linestyle='--',
                label=f"Mean = {df['FG_PCT_home'].mean():.3f}")
axes[0].set_title('Home Team FG% Distribution', fontsize=12)
axes[0].set_xlabel('Field Goal Percentage')
axes[0].legend()

# Rebounds distribution
sns.histplot(df['REB_home'], kde=True, ax=axes[1], color='darkorange', bins=40)
axes[1].axvline(df['REB_home'].mean(), color='red', linestyle='--',
                label=f"Mean = {df['REB_home'].mean():.1f}")
axes[1].set_title('Home Team Rebounds Distribution', fontsize=12)
axes[1].set_xlabel('Total Rebounds')
axes[1].legend()

# Target variable
win_counts = df['HOME_TEAM_WINS'].value_counts().sort_index()
axes[2].bar(['Away Win (0)', 'Home Win (1)'], win_counts.values,
            color=['#e74c3c', '#27ae60'], edgecolor='white', linewidth=1.5)
axes[2].set_title('Target Variable: HOME_TEAM_WINS', fontsize=12)
axes[2].set_ylabel('Count')
for i, v in enumerate(win_counts.values):
    axes[2].text(i, v + 80, f'{v:,}\n({v/len(df)*100:.1f}%)',
                 ha='center', fontweight='bold', fontsize=10)

plt.suptitle('Visualization 1: Feature & Target Distributions', fontsize=14,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Interpretation: Home FG% is approximately normally distributed around "
      f"{df['FG_PCT_home'].mean():.3f}, consistent with real NBA averages (~46%). "
      f"The target shows a moderate class imbalance: home teams win "
      f"{df['HOME_TEAM_WINS'].mean()*100:.1f}% of games. Unlike a hypothetical "
      f"74% baseline, this 58.9% rate means models must learn genuine statistical "
      f"patterns — not just predict the majority class — to achieve strong performance.")

In [ ]:
# ============================================================
# Visualization 2: Home win rate by season (Ch 3)
# ============================================================

season_stats = df.groupby('SEASON').agg(
    win_rate=('HOME_TEAM_WINS', 'mean'),
    games=('HOME_TEAM_WINS', 'count')
).reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(season_stats['SEASON'], season_stats['win_rate'],
        color='steelblue', linewidth=2.5, marker='o', markersize=7)
ax.axhline(y=df['HOME_TEAM_WINS'].mean(), color='red', linestyle='--', linewidth=1.5,
           label=f'Overall Mean = {df["HOME_TEAM_WINS"].mean():.3f}')
ax.fill_between(season_stats['SEASON'], season_stats['win_rate'],
                df['HOME_TEAM_WINS'].mean(), alpha=0.12, color='steelblue')

# Annotate COVID bubble season (2019 = 2019-20 season)
covid_row = season_stats[season_stats['SEASON'] == 2019]
if not covid_row.empty:
    ax.annotate('2020 Bubble\n(no home crowd)', xy=(2019, covid_row['win_rate'].values[0]),
                xytext=(2016, 0.46), fontsize=9, color='gray',
                arrowprops=dict(arrowstyle='->', color='gray'))

ax.set_xlabel('NBA Season', fontsize=12)
ax.set_ylabel('Home Win Rate', fontsize=12)
ax.set_ylim(0.40, 0.70)
ax.legend(fontsize=11)
ax.set_title('Visualization 2: NBA Home Win Rate by Season (2003–2022)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Interpretation: Home win rate is stable near 58-60% across most seasons, with a "
      "notable dip in 2020 when games were played in the COVID bubble without home fans. "
      "This natural experiment confirms that crowd presence drives part of home court "
      "advantage. Temporal stability justifies pooling all seasons for training.")

In [ ]:
# ============================================================
# Visualization 3: Differential features stratified by outcome (Ch 3)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# FG_PCT_DIFF distributions by outcome
wins   = df[df['HOME_TEAM_WINS'] == 1]['FG_PCT_DIFF']
losses = df[df['HOME_TEAM_WINS'] == 0]['FG_PCT_DIFF']
axes[0].hist(losses, bins=50, alpha=0.65, color='#e74c3c', density=True,
             label=f'Away Win  (mean={losses.mean():+.3f})')
axes[0].hist(wins,   bins=50, alpha=0.65, color='#27ae60', density=True,
             label=f'Home Win (mean={wins.mean():+.3f})')
axes[0].axvline(0, color='black', linestyle='--', linewidth=1.5, label='Zero')
axes[0].set_xlabel('FG% Differential (Home − Away)', fontsize=11)
axes[0].set_ylabel('Density')
axes[0].set_title('FG% Differential: Wins vs. Losses', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)

# REB_DIFF box plot by outcome
df_plot = df[['REB_DIFF', 'HOME_TEAM_WINS']].copy()
df_plot['Outcome'] = df_plot['HOME_TEAM_WINS'].map({1: 'Home Win', 0: 'Away Win'})
sns.boxplot(data=df_plot, x='Outcome', y='REB_DIFF',
            palette={'Home Win': '#27ae60', 'Away Win': '#e74c3c'}, ax=axes[1])
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_title('Rebound Differential by Outcome', fontsize=12, fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Home Rebounds − Away Rebounds', fontsize=11)

plt.suptitle('Visualization 3: Key Differentials Stratified by Outcome',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Interpretation: FG_PCT_DIFF shows clearly separated distributions by outcome — "
      "home teams that win shoot better than their opponents (positive differential mean), "
      "while losing home teams shoot worse. Rebound differential shows similar separation. "
      "Both confirm that differential features carry genuine predictive signal.")

### 2.5 Outlier Detection

In [ ]:
# ============================================================
# 2.5 Outlier detection — Tukey Fences (Ch 4)
# ============================================================

def tukey_fences(series, k=1.5):
    """Return lower and upper Tukey fences."""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - k * IQR, Q3 + k * IQR

check_cols = ['FG_PCT_home', 'FG_PCT_away', 'REB_home', 'REB_away', 'AST_home', 'AST_away']

print(f"{'Column':<20} {'Lower':>8} {'Upper':>8} {'# Out':>8} {'%':>8}")
print('-' * 60)
for col in check_cols:
    lo, hi = tukey_fences(df[col])
    n = len(df[(df[col] < lo) | (df[col] > hi)])
    print(f"{col:<20} {lo:>8.3f} {hi:>8.3f} {n:>8} {n/len(df)*100:>7.1f}%")

**Outlier strategy:**

Tukey fence analysis shows outlier rates of 0.6–0.8% per column. These are **not data errors** — they represent legitimate extreme game performances (a team shooting 60%+ from the field, grabbing 55+ rebounds in an overtime game). These are valid outcomes our model should handle in production.

**Decision: retain all observations.** Winsorizing or trimming would artificially compress extreme performances that carry real signal. Random Forest is inherently robust to outliers, and the < 1% rate means they will not distort Logistic Regression training either. No outlier treatment applied.

### 2.6 Correlations

In [ ]:
# ============================================================
# 2.6 Correlation heatmap — engineered features (Ch 3)
# ============================================================

diff_features = ['FG_PCT_DIFF', 'FG3_PCT_DIFF', 'FT_PCT_DIFF',
                 'REB_DIFF', 'AST_DIFF', 'HOME_TEAM_WINS']
corr_matrix = df[diff_features].corr()

plt.figure(figsize=(8, 6))
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, mask=mask, square=True,
            cbar_kws={'shrink': 0.8}, linewidths=0.5)
plt.title('Correlation Heatmap — Differential Features vs. Target',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

target_corr = corr_matrix['HOME_TEAM_WINS'].drop('HOME_TEAM_WINS').sort_values(ascending=False)
print('Correlations with HOME_TEAM_WINS (sorted):')
for feat, r in target_corr.items():
    print(f'  {feat:<20} r = {r:+.3f}')

**Data Quality Summary**

| Dimension | Finding |
|---|---|
| Completeness | 99 rows (0.37%) missing box-score stats — dropped via listwise deletion |
| Missingness type | MCAR — all stats missing together (suspended/postponed games) |
| Outliers | < 1% extreme values per column; all represent legitimate game performances |
| Distributions | All numeric features approximately normal; target 58.9% positive class |
| Key correlation | FG_PCT_DIFF has the strongest correlation with HOME_TEAM_WINS |
| Class balance | 58.9% home wins — moderate imbalance; ROC-AUC used as primary metric |
| Coverage | 20 NBA seasons (2003–2022), 26,552 complete games |

**Conclusion:** High-quality dataset requiring minimal cleaning. The modeling challenge is learning generalizable patterns from in-game statistics.

---
## Part 3: Model Building

### 3.1 Define Features and Target

In [ ]:
# ============================================================
# 3.1 Define features and target
# ============================================================

features = [
    # Raw home team statistics
    'FG_PCT_home', 'FG3_PCT_home', 'FT_PCT_home', 'AST_home', 'REB_home',
    # Raw away team statistics
    'FG_PCT_away', 'FG3_PCT_away', 'FT_PCT_away', 'AST_away', 'REB_away',
    # Engineered differential features
    'FG_PCT_DIFF', 'FG3_PCT_DIFF', 'FT_PCT_DIFF', 'REB_DIFF', 'AST_DIFF'
]

X = df[features]
y = df['HOME_TEAM_WINS']

print(f'Feature matrix:  {X.shape}')
print(f'Target classes:  {dict(y.value_counts().sort_index())}')
print(f'Naive baseline (always predict home win): {y.mean():.4f}')

### 3.2 Train/Test Split

In [ ]:
# ============================================================
# 3.2 Train/test split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y        # preserves 58.9% class balance in both splits
)

print(f'Training set:   {X_train.shape[0]:,} games')
print(f'Test set:       {X_test.shape[0]:,} games')
print(f'Train win rate: {y_train.mean():.4f}')
print(f'Test win rate:  {y_test.mean():.4f}')
print('Stratification confirmed — class balance preserved in both splits.')

### 3.3 Model 1: Logistic Regression (Baseline)

In [ ]:
# ============================================================
# 3.3 Model 1: Logistic Regression (Ch 9)
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

model_1 = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

model_1.fit(X_train, y_train)
y_pred_1 = model_1.predict(X_test)
y_prob_1 = model_1.predict_proba(X_test)[:, 1]

print('=== Model 1: Logistic Regression ===')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_1):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_1):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred_1):.4f}')
print(f'F1 Score:  {f1_score(y_test, y_pred_1):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_prob_1):.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred_1, target_names=['Away Win', 'Home Win']))

### 3.3b Model 2: Random Forest

In [ ]:
# ============================================================
# 3.3b Model 2: Random Forest (Ch 18)
# ============================================================

from sklearn.ensemble import RandomForestClassifier

model_2 = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

model_2.fit(X_train, y_train)
y_pred_2 = model_2.predict(X_test)
y_prob_2 = model_2.predict_proba(X_test)[:, 1]

print('=== Model 2: Random Forest ===')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_2):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_2):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred_2):.4f}')
print(f'F1 Score:  {f1_score(y_test, y_pred_2):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_prob_2):.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred_2, target_names=['Away Win', 'Home Win']))

### 3.4 Cross-Validation Comparison

In [ ]:
# ============================================================
# 3.4 5-fold cross-validation comparison (Ch 15)
# ============================================================

scoring = 'accuracy'

cv_1 = cross_val_score(model_1, X_train, y_train, cv=5, scoring=scoring)
cv_2 = cross_val_score(model_2, X_train, y_train, cv=5, scoring=scoring)

print(f'Model 1 (Logistic Regression) CV {scoring}: {cv_1.mean():.4f} +/- {cv_1.std():.4f}')
print(f'Model 2 (Random Forest)       CV {scoring}: {cv_2.mean():.4f} +/- {cv_2.std():.4f}')

comparison = pd.DataFrame({
    'Model': ['Model 1: Logistic Regression', 'Model 2: Random Forest'],
    'Test Accuracy':  [accuracy_score(y_test, y_pred_1),  accuracy_score(y_test, y_pred_2)],
    'Test Precision': [precision_score(y_test, y_pred_1), precision_score(y_test, y_pred_2)],
    'Test Recall':    [recall_score(y_test, y_pred_1),    recall_score(y_test, y_pred_2)],
    'Test F1':        [f1_score(y_test, y_pred_1),        f1_score(y_test, y_pred_2)],
    'ROC-AUC':        [roc_auc_score(y_test, y_prob_1),   roc_auc_score(y_test, y_prob_2)],
    'CV Mean':        [cv_1.mean(), cv_2.mean()],
    'CV Std':         [cv_1.std(),  cv_2.std()],
})
comparison.set_index('Model').round(4)

---
## Part 4: Feature Importance + Visualization

### 4.1 Feature Importance

In [ ]:
# ============================================================
# 4.1 Feature importance — Random Forest Gini (Ch 19)
# ============================================================

importances = pd.Series(
    model_2.feature_importances_, index=features
).sort_values(ascending=True)

colors = ['#e74c3c' if i >= len(importances) - 5 else 'steelblue'
          for i in range(len(importances))]

fig, ax = plt.subplots(figsize=(9, 6))
importances.plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('Feature Importance (Gini Impurity Reduction)', fontsize=12)
ax.set_title('Random Forest Feature Importance', fontsize=13, fontweight='bold')

# Required caveat banner (Ch 19)
ax.text(
    0.98, 0.02,
    'Predictive importance only.\nDoes not imply causal effect.',
    transform=ax.transAxes, fontsize=9, ha='right', va='bottom',
    style='italic', color='#c0392b',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='#fdedec', edgecolor='#e74c3c')
)

plt.tight_layout()
plt.show()

print('Top 5 predictors:')
print(importances.tail(5).sort_values(ascending=False).to_string())

### 4.2 Key Visualization for Report

In [ ]:
# ============================================================
# 4.2 Key visualization: ROC curves + confusion matrix
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC curves
for prob, name, color in [
    (y_prob_1, 'Logistic Regression', 'steelblue'),
    (y_prob_2, 'Random Forest',       'darkorange')
]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    axes[0].plot(fpr, tpr, color=color, lw=2.5, label=f'{name} (AUC = {auc:.3f})')

axes[0].plot([0,1],[0,1], 'k--', lw=1.5, label='Random Baseline (AUC = 0.500)')
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title('ROC Curves — Model Comparison', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].set_xlim([0,1]); axes[0].set_ylim([0,1.02])

# Confusion matrix for recommended model (Logistic Regression)
cm = confusion_matrix(y_test, y_pred_1)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Away Win', 'Home Win'],
            yticklabels=['Away Win', 'Home Win'],
            linewidths=1, linecolor='white', cbar_kws={'shrink': 0.8})
axes[1].set_xlabel('Predicted', fontsize=12)
axes[1].set_ylabel('Actual', fontsize=12)
axes[1].set_title('Confusion Matrix — Logistic Regression (Test Set)',
                  fontsize=13, fontweight='bold')

plt.suptitle(
    'Key Finding: Both models achieve ~84% accuracy and 0.92 AUC — '
    'well above the 58.9% naive baseline',
    fontsize=11, style='italic', y=1.02)
plt.tight_layout()
plt.show()

---
## Part 5: Recommendation

**Situation:** A sports analytics firm needs real-time win-probability estimates for NBA games using live in-game statistics. The dataset contains 26,552 NBA games from 2003–2022 from the official Kaggle NBA Games dataset (nathanlauga). The target variable is `HOME_TEAM_WINS` (binary), with home teams winning 58.9% of games — a meaningful but not extreme class imbalance.

**Complication:** The 2020 COVID bubble season (games played without home fans) shows a sharp dip in home win rate, confirming that home court advantage is partially crowd-driven and context-dependent — not a fixed statistical property. Our features are recorded *during* the game being predicted, making this a descriptive rather than purely pre-game forecasting pipeline. The key question is whether statistical patterns learned from 20 seasons generalize reliably to unseen games.

**Resolution:** Both models substantially outperform the naive baseline. **Logistic Regression is the recommended model**: it achieves **83.7% test accuracy** and **0.923 ROC-AUC** — 24.8 percentage points above the baseline — while providing calibrated probability estimates and interpretable coefficients that explain each prediction to the analytics team. The dominant predictor is `FG_PCT_DIFF` (Random Forest importance: 32.9%), confirming field goal efficiency gap as the single strongest signal in game outcomes.

**Uncertainty Statement:** Based on our cross-validation results (accuracy = 0.8422 +/- 0.0056), we estimate the model correctly predicts game outcomes in approximately 84% of cases. The CV standard deviation of 0.006 confirms stable generalization with no overfitting. The primary limitation is that in-game statistics are collected *during* the game — a pre-game production system should replace these with season-to-date rolling averages, opponent-adjusted efficiency ratings, and contextual variables (rest days, travel distance). We recommend deploying the Logistic Regression model with the caveat that probability estimates should be recalibrated annually as NBA playing styles evolve.

---
## Part 6: Streamlit Export Guide

### 6.1 Deployment Steps

1. Save trained model: `joblib.dump(model_1, 'model.pkl')`
2. Push to GitHub: `app.py`, `games.csv`, `requirements.txt`, `model.pkl`
3. Go to [streamlit.io/cloud](https://streamlit.io/cloud) → New app → connect repo → set main file to `app.py`
4. Submit permanent URL on Canvas

### 6.2 requirements.txt
```
streamlit>=1.32.0
pandas>=2.0.0
numpy>=1.24.0
scikit-learn>=1.3.0
matplotlib>=3.7.0
seaborn>=0.12.0
joblib>=1.3.0
```

In [ ]:
# ============================================================
# 6.3 Save the trained model
# ============================================================

import joblib

joblib.dump(model_1, 'model.pkl')
print('model.pkl saved — Logistic Regression (recommended model)')
print('Include model.pkl in your GitHub repo alongside app.py and games.csv')

---
## Part 7: AI Methodology Appendix

All AI co-pilot interactions documented using the **P.R.I.M.E.** framework (Prep → Request → Iterate → Mechanism Check → Evaluate). Worth 10% of project grade.

---

### AI Interaction 1 — Dataset Selection

**Prep:** Needed a sports classification dataset with 1,000+ rows, public URL, binary target. No dataset chosen — starting from scratch.
> Context given to AI: course requirements (1,000+ rows, public URL, binary target), domain preference (sports), problem type (classification).

**Request:** "Help me choose a sports classification dataset for a capstone ML project. I need 1,000+ rows, a binary prediction target, a public Kaggle or UCI source URL, and a meaningful sports analytics stakeholder."

**Iterate:** AI suggested the NBA Games dataset (Kaggle — nathanlauga) with `HOME_TEAM_WINS` as the binary target. Accepted in one iteration — dataset met all requirements.

**Mechanism Check:** Verified on Kaggle: 26,651 rows, binary target, public download. Confirmed home win rate (~59%) was plausible by cross-referencing published NBA home court advantage statistics.

**Evaluate:** Accepted the dataset. I independently decided to use only `games.csv` as the primary modeling file (rather than joining player-level `games_details.csv`), reasoning that game-level aggregates are cleaner for an initial model. Stakeholder framing (sports analytics firm) was my own addition.

---

### AI Interaction 2 — Missing Data Classification

**Prep:** After loading `games.csv`, discovered 99 rows where all 12 box-score columns are missing simultaneously. Needed to classify under the Ch. 1 MCAR/MAR/MNAR framework and choose a handling strategy.

**Request:** "In my NBA games dataset, 99 rows have all 12 box-score statistic columns missing simultaneously, but GAME_ID and HOME_TEAM_WINS are present. Under the MCAR/MAR/MNAR framework, how should I classify this and what strategy should I use?"

**Iterate:** AI classified as MCAR (missingness due to game suspension/postponement, unrelated to stats or outcome) and recommended listwise deletion. Accepted in one iteration.

**Mechanism Check:** Checked that rows with missing stats had valid `HOME_TEAM_WINS` values (confirming game was recorded but not completed), and that the 0.37% drop rate would not materially affect class balance (verified: train and test win rates stayed at 0.589 post-drop).

**Evaluate:** Accepted MCAR classification and listwise deletion. I independently verified the 0.37% drop rate was too small to bias the training distribution before accepting the recommendation.

---

### AI Interaction 3 — Model Recommendation and SCR Resolution

**Prep:** Both models achieved ~84% accuracy and ~0.92 AUC on the test set. Needed to write the SCR Resolution section recommending one model with empirical justification.

**Request:** "My Logistic Regression gets 83.7% accuracy and 0.923 AUC, and my Random Forest gets 83.4% and 0.921 AUC on NBA game outcome prediction. Help me write an SCR Resolution paragraph recommending one model and explaining the trade-off to a sports analytics stakeholder."

**Iterate:** First draft recommended Random Forest, citing slightly higher training CV accuracy. I pushed back asking to prioritize interpretability for the analytics use case — second draft correctly recommended Logistic Regression for calibrated probabilities and explainability. Two iterations.

**Mechanism Check:** Verified that Logistic Regression's held-out test AUC (0.9226) was indeed higher than Random Forest (0.9206) on the test set, confirming the recommendation was empirically — not just intuitively — justified.

**Evaluate:** Accepted the revised recommendation. I independently added the COVID bubble observation (crowd absence drove win rate drop), the caveat about in-game vs. pre-game features, and the specific CV numbers (0.8422 ± 0.0056) — none of these were in the AI's draft.